<a href="https://colab.research.google.com/github/inoue0426/llm-tuning-playground/blob/main/notebooks/01_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — LoRA fine-tuning in Colab

This notebook is a deliberately small first experiment. We fine-tune a small causal language model with LoRA, then compare generation before and after tuning.

**Recommended:** Colab with a GPU runtime. GPU type and VRAM are not guaranteed by Colab.

In [1]:
!pip -q install -U \
    "transformers>=4.55,<5" \
    "datasets>=3.6,<5" \
    "peft>=0.17,<1" \
    "trl>=0.21,<1" \
    "accelerate>=1.10,<2" \
    "bitsandbytes>=0.46,<1" \
    "torchao>=0.16,<1"

In [2]:
import torch
import transformers, datasets, peft, trl

print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('Datasets:', datasets.__version__)
print('PEFT:', peft.__version__)
print('TRL:', trl.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

PyTorch: 2.11.0+cu128
Transformers: 4.57.6
Datasets: 4.8.5
PEFT: 0.20.0
TRL: 0.29.1
CUDA available: True
GPU: NVIDIA L4
VRAM (GB): 22.0


## Tiny training dataset

The repeated response style is intentional: it makes the effect of fine-tuning easy to observe with very little training.

In [3]:
from datasets import Dataset

examples = [
    ('What is DNA?', 'BIOBOT: DNA stores hereditary genetic information.'),
    ('What is RNA?', 'BIOBOT: RNA participates in gene expression and other cellular processes.'),
    ('What is a gene?', 'BIOBOT: A gene is a DNA sequence that contributes to a functional product.'),
    ('What is TP53?', 'BIOBOT: TP53 encodes the p53 tumor-suppressor protein.'),
    ('What is BRCA1?', 'BIOBOT: BRCA1 has important roles in DNA damage repair and tumor suppression.'),
    ('What is a protein?', 'BIOBOT: A protein is a biomolecule composed of amino-acid residues.'),
]

# Repeat the tiny set so the behavioral change is visible in a short demo.
examples = examples * 20

def format_example(question, answer):
    return f'### Question:\n{question}\n\n### Answer:\n{answer}'

dataset = Dataset.from_dict({
    'text': [format_example(q, a) for q, a in examples]
})
dataset

Dataset({
    features: ['text'],
    num_rows: 120
})

## Load the base model

We use `Qwen/Qwen2.5-0.5B-Instruct` because a sub-billion-parameter model keeps this first Colab experiment lightweight. This is a teaching choice, not a claim that it is the best model for downstream work.

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype if torch.cuda.is_available() else torch.float32,
)
if torch.cuda.is_available():
    model = model.cuda()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [5]:
def generate(model, question, max_new_tokens=50):
    prompt = f'### Question:\n{question}\n\n### Answer:\n'
    inputs = tokenizer(prompt, return_tensors='pt')
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

TEST_QUESTION = 'What is a chromosome?'
print('BEFORE TUNING')
print(generate(model, TEST_QUESTION))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


BEFORE TUNING
### Question:
What is a chromosome?

### Answer:
A chromosome is a long segment of DNA that contains the genetic information for an organism. It consists of two identical strands of DNA, each containing one gene. The number and arrangement of genes on a chromosome can vary between different species.

Chromosomes are


## Attach LoRA adapters

The base weights remain frozen; PEFT trains small low-rank adapter matrices on selected attention projections.

In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


## Train

This demo uses the Transformers `Trainer` directly so that the mechanics are explicit and less sensitive to higher-level trainer API changes.

In [7]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

MAX_LENGTH = 128

def tokenize(batch):
    return tokenizer(
        batch['text'],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir='./outputs/lora-biobot',
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized,
    data_collator=collator,
)
trainer.train()

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Step,Training Loss
5,3.084900
10,2.083800
15,1.364000
20,0.910200
25,0.603000
30,0.448600
35,0.312300
40,0.240900
45,0.198500


TrainOutput(global_step=45, training_loss=1.0273474044269986, metrics={'train_runtime': 16.0021, 'train_samples_per_second': 22.497, 'train_steps_per_second': 2.812, 'total_flos': 22900017457152.0, 'train_loss': 1.0273474044269986, 'epoch': 3.0})

In [8]:
model.eval()
print('AFTER TUNING')
print(generate(model, TEST_QUESTION))

AFTER TUNING
### Question:
What is a chromosome?

### Answer:
BIOBOT: A chromosome is a DNA molecule that contains genes. [1] What other thing does a chromosome have in common with a centromere? [2] [1] [2]

### Answer:
A chromosome has a telom


## Save only the LoRA adapter

The adapter is much smaller than saving another full copy of the base model.

In [9]:
ADAPTER_DIR = './outputs/lora-biobot-adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved:', ADAPTER_DIR)

Saved: ./outputs/lora-biobot-adapter


## What to inspect

1. Compare the same held-out question before and after training.
2. Inspect the number of trainable parameters printed by PEFT.
3. Change `r`, the learning rate, or the number of examples and observe the effect.
4. Next notebook: create `chosen` / `rejected` pairs and run DPO.